# 02 - Probabilistic Baseline Models

This notebook performs the first classification experiment of Practical
Application 2 using all valid predictor variables.

The objective is to establish a probabilistic baseline before applying the
different feature-selection strategies.

The following probabilistic classifier families studied in the course are
evaluated:

- Mixed Naive Bayes.
- Logistic Regression.
- Linear Discriminant Analysis (LDA).
- Quadratic Discriminant Analysis (QDA).

All models are evaluated using the same stratified cross-validation folds and
the same performance metrics.

In [15]:
import pandas as pd
import numpy as np

from scipy.special import logsumexp

from sklearn.base import (
    BaseEstimator,
    ClassifierMixin,
    clone
)

from sklearn.compose import ColumnTransformer

from sklearn.preprocessing import (
    OneHotEncoder,
    OrdinalEncoder,
    StandardScaler
)

from sklearn.pipeline import Pipeline

from sklearn.model_selection import StratifiedKFold

from sklearn.linear_model import LogisticRegression

from sklearn.discriminant_analysis import (
    LinearDiscriminantAnalysis,
    QuadraticDiscriminantAnalysis
)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

## 1. Dataset Preparation

The same dataset preparation defined in the previous notebook is used.

The identifier and constant variables are removed and `Attrition` is encoded
as a binary target:

- `No` -> 0
- `Yes` -> 1

No feature selection is performed in this experiment.

In [16]:
DATA_PATH = "../../data/raw/WA_Fn-UseC_-HR-Employee-Attrition.csv"

df = pd.read_csv(DATA_PATH)

columns_to_remove = [
    "EmployeeNumber",
    "EmployeeCount",
    "Over18",
    "StandardHours"
]

df_model = df.drop(columns=columns_to_remove)

X = df_model.drop(columns=["Attrition"])

y = df_model["Attrition"].map({
    "No": 0,
    "Yes": 1
})

print("Predictor matrix shape:", X.shape)
print("Target shape:", y.shape)

Predictor matrix shape: (1470, 30)
Target shape: (1470,)


## 2. Predictor Types

The same distinction between continuous, ordinal, and categorical predictors
defined in the data-preparation notebook is maintained.

For Mixed Naive Bayes, ordinal and nominal categorical variables are treated
as discrete predictors, while the remaining numerical variables are treated as
continuous predictors.

The other classifiers require a completely numerical representation.

In [17]:
ordinal_features = [
    "Education",
    "EnvironmentSatisfaction",
    "JobInvolvement",
    "JobLevel",
    "JobSatisfaction",
    "PerformanceRating",
    "RelationshipSatisfaction",
    "StockOptionLevel",
    "WorkLifeBalance"
]

numerical_features = (
    X.select_dtypes(include=np.number)
    .columns
    .tolist()
)

categorical_features = (
    X.select_dtypes(exclude=np.number)
    .columns
    .tolist()
)

continuous_features = [
    feature
    for feature in numerical_features
    if feature not in ordinal_features
]

discrete_features = (
    ordinal_features
    + categorical_features
)

print("Continuous predictors:", len(continuous_features))
print("Discrete predictors:", len(discrete_features))
print("Total:", len(continuous_features) + len(discrete_features))

Continuous predictors: 14
Discrete predictors: 16
Total: 30


## 3. Mixed Naive Bayes

The course presents a Naive Bayes classifier that combines discrete and
continuous predictors.

For continuous predictors, class-conditional Gaussian distributions are
estimated.

For discrete predictors, class-conditional category probabilities are
estimated using Laplace smoothing.

The Naive Bayes conditional-independence assumption is maintained:

\[
P(C \mid X)
\propto
P(C)
\prod_j P(X_j \mid C)
\]

The implementation below follows this mixed discrete/continuous formulation
and provides posterior class probabilities required for probabilistic
evaluation.

In [18]:
class MixedNaiveBayes(BaseEstimator, ClassifierMixin):
    """
    Naive Bayes classifier for mixed continuous and discrete predictors.

    Continuous predictors are modeled with class-conditional Gaussian
    distributions.

    Discrete predictors are modeled using class-conditional frequency
    estimates with Laplace smoothing.
    """

    def __init__(
        self,
        continuous_features,
        discrete_features,
        alpha=1.0,
        var_smoothing=1e-9
    ):
        self.continuous_features = continuous_features
        self.discrete_features = discrete_features
        self.alpha = alpha
        self.var_smoothing = var_smoothing

    def fit(self, X, y):

        X = X.copy()
        y = np.asarray(y)

        self.classes_ = np.unique(y)

        self.class_prior_ = {}
        self.gaussian_parameters_ = {}
        self.discrete_probabilities_ = {}
        self.discrete_default_probabilities_ = {}

        n_samples = len(y)

        for cls in self.classes_:

            class_mask = (y == cls)
            X_cls = X.loc[class_mask]

            class_count = class_mask.sum()

            self.class_prior_[cls] = (
                class_count / n_samples
            )

            # Continuous predictors
            self.gaussian_parameters_[cls] = {}

            for feature in self.continuous_features:

                values = (
                    X_cls[feature]
                    .astype(float)
                    .to_numpy()
                )

                mean = values.mean()
                variance = values.var()

                variance += self.var_smoothing

                self.gaussian_parameters_[cls][feature] = (
                    mean,
                    variance
                )

            # Discrete predictors
            self.discrete_probabilities_[cls] = {}
            self.discrete_default_probabilities_[cls] = {}

            for feature in self.discrete_features:

                all_categories = (
                    X[feature]
                    .astype(str)
                    .unique()
                )

                class_values = (
                    X_cls[feature]
                    .astype(str)
                )

                counts = (
                    class_values
                    .value_counts()
                )

                # An additional probability mass is reserved for
                # categories that were not observed in the training fold.
                n_categories = len(all_categories) + 1

                denominator = (
                    class_count
                    + self.alpha * n_categories
                )

                probabilities = {}

                for category in all_categories:

                    probabilities[category] = (
                        counts.get(category, 0)
                        + self.alpha
                    ) / denominator

                self.discrete_probabilities_[cls][feature] = (
                    probabilities
                )

                self.discrete_default_probabilities_[cls][feature] = (
                    self.alpha / denominator
                )

        return self

    def _joint_log_likelihood(self, X):

        X = X.copy()

        joint_log_likelihood = []

        for cls in self.classes_:

            log_probability = np.full(
                X.shape[0],
                np.log(self.class_prior_[cls]),
                dtype=float
            )

            # Gaussian contribution
            for feature in self.continuous_features:

                mean, variance = (
                    self.gaussian_parameters_[cls][feature]
                )

                values = (
                    X[feature]
                    .astype(float)
                    .to_numpy()
                )

                log_probability += (
                    -0.5 * np.log(
                        2 * np.pi * variance
                    )
                    - (
                        (values - mean) ** 2
                        / (2 * variance)
                    )
                )

            # Discrete contribution
            for feature in self.discrete_features:

                probabilities = (
                    self.discrete_probabilities_[cls][feature]
                )

                default_probability = (
                    self.discrete_default_probabilities_[cls][feature]
                )

                values = (
                    X[feature]
                    .astype(str)
                    .to_numpy()
                )

                feature_probabilities = np.array([
                    probabilities.get(
                        value,
                        default_probability
                    )
                    for value in values
                ])

                log_probability += np.log(
                    feature_probabilities
                )

            joint_log_likelihood.append(
                log_probability
            )

        return np.column_stack(
            joint_log_likelihood
        )

    def predict_proba(self, X):

        joint_log_likelihood = (
            self._joint_log_likelihood(X)
        )

        log_normalizer = logsumexp(
            joint_log_likelihood,
            axis=1,
            keepdims=True
        )

        return np.exp(
            joint_log_likelihood
            - log_normalizer
        )

    def predict(self, X):

        probabilities = self.predict_proba(X)

        return self.classes_[
            np.argmax(
                probabilities,
                axis=1
            )
        ]

## 4. Numerical Representation for Logistic Regression, LDA and QDA

The probabilistic classifiers do not necessarily require the same numerical
representation.

### Logistic Regression

Continuous and ordinal predictors are standardized.

Nominal categorical predictors are one-hot encoded, using one reference
category to avoid perfect dummy-variable dependence.

### Linear and Quadratic Discriminant Analysis

LDA and QDA estimate covariance structures. Expanding nominal variables into
multiple dummy variables can introduce strong dependencies and rank-deficient
covariance matrices, especially for the minority attrition class.

For discriminant analysis, categorical predictors are therefore represented
with a single numerical code per original variable rather than expanded into
multiple dummy columns.

This preserves the original number of predictors and avoids unnecessary dummy
redundancy.

The numerical encoding of nominal attributes is an implementation choice
required to provide a numerical representation to LDA and QDA; it should not
be interpreted as implying a meaningful ordinal relationship between nominal
categories.

All preprocessing transformations are fitted exclusively using the training
portion of each cross-validation fold.

In [19]:
scaled_numeric_features = (
    continuous_features
    + ordinal_features
)

# Logistic Regression:
# scale numerical variables and one-hot encode nominal predictors.
logistic_preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            StandardScaler(),
            scaled_numeric_features
        ),
        (
            "cat",
            OneHotEncoder(
                drop="first",
                handle_unknown="ignore",
                sparse_output=False
            ),
            categorical_features
        )
    ]
)

# LDA / QDA:
# keep one numerical column per original predictor.
discriminant_preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            StandardScaler(),
            scaled_numeric_features
        ),
        (
            "cat",
            OrdinalEncoder(
                handle_unknown="use_encoded_value",
                unknown_value=-1
            ),
            categorical_features
        )
    ]
)

## 5. Probabilistic Classification Algorithms

One representative implementation is used for each main probabilistic
classifier family selected from the course material.

The classifiers are:

- Mixed Naive Bayes.
- Logistic Regression.
- Linear Discriminant Analysis.
- Quadratic Discriminant Analysis.

The objective is to compare classifier families, not every individual variant
mentioned in the lecture slides.

No hyperparameter optimization is performed at this stage.

### Representation Dimensionality

Logistic Regression uses one-hot encoding for nominal attributes, whereas LDA
and QDA retain one numerical column per original predictor.

The dimensionality of both representations is checked below before model
evaluation.

In [20]:
X_logistic_preview = (
    logistic_preprocessor
    .fit_transform(X)
)

X_discriminant_preview = (
    discriminant_preprocessor
    .fit_transform(X)
)

print(
    "Logistic Regression representation:",
    X_logistic_preview.shape
)

print(
    "LDA/QDA representation:",
    X_discriminant_preview.shape
)

Logistic Regression representation: (1470, 44)
LDA/QDA representation: (1470, 30)


In [21]:
mixed_nb = MixedNaiveBayes(
    continuous_features=continuous_features,
    discrete_features=discrete_features,
    alpha=1.0
)

logistic_pipeline = Pipeline(
    steps=[
        (
            "preprocessing",
            clone(logistic_preprocessor)
        ),
        (
            "classifier",
            LogisticRegression(
                max_iter=2000,
                random_state=42
            )
        )
    ]
)

lda_pipeline = Pipeline(
    steps=[
        (
            "preprocessing",
            clone(discriminant_preprocessor)
        ),
        (
            "classifier",
            LinearDiscriminantAnalysis()
        )
    ]
)

qda_pipeline = Pipeline(
    steps=[
        (
            "preprocessing",
            clone(discriminant_preprocessor)
        ),
        (
            "classifier",
            QuadraticDiscriminantAnalysis()
        )
    ]
)

## 6. Evaluation Strategy

Model performance is estimated using stratified 10-fold cross-validation.

Stratification approximately preserves the proportion of attrition and
non-attrition observations in every fold, which is appropriate for the
imbalanced target distribution.

All model-specific preprocessing is learned exclusively from the training
portion of each fold.

The same folds are used for all classifiers to make their results directly
comparable.

In [22]:
cv = StratifiedKFold(
    n_splits=10,
    shuffle=True,
    random_state=42
)

## 7. Evaluation Metrics

The models are evaluated using classification and probability-based measures
covered in the course.

### Classification metrics

- **Accuracy:** proportion of correctly classified observations.
- **Sensitivity / Recall:** proportion of actual attrition cases correctly
  detected.
- **Specificity:** proportion of non-attrition cases correctly detected.
- **Positive Predictive Value / Precision:** proportion of predicted attrition
  cases that actually correspond to attrition.
- **F1-measure:** harmonic mean of Precision and Recall.

### Probability-based metrics

- **AUC:** summarizes the ability of the classifier to rank positive cases
  above negative cases across classification thresholds.
- **Brier score:** measures the squared distance between predicted class
  probabilities and the true class indicator vector.

Lower Brier scores indicate better probabilistic predictions.

In [23]:
def multiclass_brier_score(
    y_true,
    probabilities,
    classes
):
    """
    Compute the mean squared Euclidean distance between
    the predicted probability vector and the true
    class-indicator vector.
    """

    y_true = np.asarray(y_true)

    indicator_matrix = np.zeros_like(
        probabilities,
        dtype=float
    )

    for index, cls in enumerate(classes):

        indicator_matrix[:, index] = (
            y_true == cls
        ).astype(float)

    squared_error = (
        probabilities
        - indicator_matrix
    ) ** 2

    return np.mean(
        squared_error.sum(axis=1)
    )

## 8. Common Evaluation Function

A custom cross-validation evaluation function is used so that all
classification metrics and the full probability-vector Brier score can be
calculated consistently.

For every fold:

1. The model is fitted using the training observations.
2. Class labels are predicted for the validation observations.
3. Posterior class probabilities are predicted.
4. Classification and probability-based metrics are calculated.
5. The results are stored for later aggregation.

In [24]:
def evaluate_probabilistic_model(
    name,
    estimator,
    X,
    y,
    cv
):
    fold_results = []

    for fold, (train_index, test_index) in enumerate(
        cv.split(X, y),
        start=1
    ):

        X_train = X.iloc[train_index].copy()
        X_test = X.iloc[test_index].copy()

        y_train = y.iloc[train_index].copy()
        y_test = y.iloc[test_index].copy()

        model = clone(estimator)

        model.fit(
            X_train,
            y_train
        )

        y_pred = model.predict(X_test)

        probabilities = model.predict_proba(
            X_test
        )

        classes = model.classes_

        positive_index = np.where(
            classes == 1
        )[0][0]

        positive_probability = (
            probabilities[:, positive_index]
        )

        fold_result = {
            "Model": name,
            "Fold": fold,

            "Accuracy": accuracy_score(
                y_test,
                y_pred
            ),

            "Precision": precision_score(
                y_test,
                y_pred,
                zero_division=0
            ),

            "Recall": recall_score(
                y_test,
                y_pred,
                pos_label=1
            ),

            "Specificity": recall_score(
                y_test,
                y_pred,
                pos_label=0
            ),

            "F1": f1_score(
                y_test,
                y_pred,
                zero_division=0
            ),

            "AUC": roc_auc_score(
                y_test,
                positive_probability
            ),

            "Brier": multiclass_brier_score(
                y_test,
                probabilities,
                classes
            )
        }

        fold_results.append(
            fold_result
        )

    return pd.DataFrame(
        fold_results
    )

In [25]:
models = {
    "Mixed Naive Bayes": mixed_nb,
    "Logistic Regression": logistic_pipeline,
    "LDA": lda_pipeline,
    "QDA": qda_pipeline
}

## 9. Model Evaluation

Each probabilistic classifier is evaluated using the complete set of valid
predictors and the same stratified ten-fold partitions.

No feature-selection method is applied in this baseline experiment.

In [26]:
all_fold_results = []

for name, estimator in models.items():

    print(f"Evaluating {name}...")

    model_results = evaluate_probabilistic_model(
        name=name,
        estimator=estimator,
        X=X,
        y=y,
        cv=cv
    )

    all_fold_results.append(
        model_results
    )

print("\nEvaluation completed.")

Evaluating Mixed Naive Bayes...
Evaluating Logistic Regression...
Evaluating LDA...
Evaluating QDA...

Evaluation completed.


In [27]:
baseline_fold_results = pd.concat(
    all_fold_results,
    ignore_index=True
)

display(
    baseline_fold_results.round(4)
)

,Model,Fold,Accuracy,Precision,Recall,Specificity,F1,AUC,Brier
0,Mixed Naive Bayes,1,0.7755,0.3902,0.6667,0.7967,0.4923,0.8062,0.3186
1,Mixed Naive Bayes,2,0.7959,0.4118,0.5833,0.8374,0.4828,0.7212,0.3268
2,Mixed Naive Bayes,3,0.7551,0.3500,0.5833,0.7886,0.4375,0.6762,0.3611
3,Mixed Naive Bayes,4,0.7823,0.4048,0.7083,0.7967,0.5152,0.8127,0.2919
4,Mixed Naive Bayes,5,0.8231,0.4667,0.5833,0.8699,0.5185,0.8391,0.2361
5,Mixed Naive Bayes,6,0.8095,0.4412,0.6250,0.8455,0.5172,0.7622,0.3016
6,Mixed Naive Bayes,7,0.8571,0.5405,0.8333,0.8618,0.6557,0.9214,0.1869
7,Mixed Naive Bayes,8,0.7755,0.3529,0.5217,0.8226,0.4211,0.7458,0.3066
8,Mixed Naive Bayes,9,0.8163,0.4444,0.6957,0.8387,0.5424,0.7738,0.2939
9,Mixed Naive Bayes,10,0.7959,0.4054,0.6522,0.8226,0.5000,0.7924,0.2788


## 10. Cross-Validation Summary

For every classifier, the mean and standard deviation of each metric are
calculated across the ten validation folds.

The mean represents average predictive performance, while the standard
deviation provides information about variability across folds.

In [28]:
metrics = [
    "Accuracy",
    "Precision",
    "Recall",
    "Specificity",
    "F1",
    "AUC",
    "Brier"
]

summary_rows = []

for model_name in baseline_fold_results["Model"].unique():

    model_data = baseline_fold_results[
        baseline_fold_results["Model"] == model_name
    ]

    row = {
        "Model": model_name
    }

    for metric in metrics:

        row[f"{metric}_mean"] = model_data[metric].mean()
        row[f"{metric}_std"] = model_data[metric].std()

    summary_rows.append(row)

baseline_results = pd.DataFrame(summary_rows)

display(
    baseline_results.round(4)
)

,Model,Accuracy_mean,Accuracy_std,Precision_mean,Precision_std,Recall_mean,Recall_std,Specificity_mean,Specificity_std,F1_mean,F1_std,AUC_mean,AUC_std,Brier_mean,Brier_std
0,Mixed Naive Bayes,0.7986,0.0293,0.4208,0.0563,0.6453,0.0879,0.8281,0.0278,0.5083,0.0639,0.7851,0.0674,0.2902,0.0487
1,Logistic Regression,0.8823,0.0184,0.7344,0.1076,0.4259,0.0703,0.9700,0.0127,0.5373,0.0781,0.8375,0.0393,0.1857,0.0263
2,LDA,0.8639,0.0280,0.6740,0.1933,0.3168,0.1049,0.9692,0.0194,0.4256,0.1247,0.8146,0.0394,0.2036,0.0285
3,QDA,0.8150,0.0319,0.4184,0.1022,0.3755,0.1034,0.8994,0.0235,0.3946,0.1003,0.7211,0.0336,0.2786,0.0256


## 11. Baseline Performance Summary

The following table contains the mean values used for the main comparison.

Because PA2 evaluates probabilistic classifiers, particular attention is given
not only to F1-measure and Recall but also to AUC and Brier score.

Higher values are better for Accuracy, Precision, Recall, Specificity, F1 and
AUC.

Lower values are better for Brier score.

In [29]:
baseline_summary = baseline_results[
    [
        "Model",
        "Accuracy_mean",
        "Precision_mean",
        "Recall_mean",
        "Specificity_mean",
        "F1_mean",
        "AUC_mean",
        "Brier_mean"
    ]
].copy()

baseline_summary.columns = [
    "Model",
    "Accuracy",
    "Precision",
    "Recall",
    "Specificity",
    "F1",
    "AUC",
    "Brier"
]

baseline_summary = baseline_summary.sort_values(
    "F1",
    ascending=False
)

display(
    baseline_summary.round(4)
)

,Model,Accuracy,Precision,Recall,Specificity,F1,AUC,Brier
1,Logistic Regression,0.8823,0.7344,0.4259,0.9700,0.5373,0.8375,0.1857
0,Mixed Naive Bayes,0.7986,0.4208,0.6453,0.8281,0.5083,0.7851,0.2902
2,LDA,0.8639,0.6740,0.3168,0.9692,0.4256,0.8146,0.2036
3,QDA,0.8150,0.4184,0.3755,0.8994,0.3946,0.7211,0.2786


## 12. Numerical Stability Check

LDA and QDA were successfully evaluated using the reduced numerical
representation with one encoded column per original nominal predictor.

No numerical warnings or fitting errors were produced during the final
cross-validation experiment.

This confirms that avoiding the expanded dummy representation was sufficient
to obtain stable covariance estimates without introducing additional
regularization.

## 13. Best Observed Baseline Results

The probabilistic baseline provides the reference performance obtained using
all valid predictor variables.

The main aspects to inspect are:

- Which classifier obtains the highest F1-measure.
- Which classifier provides the highest Recall.
- Which classifier achieves the highest AUC.
- Which classifier obtains the lowest Brier score.
- Whether high Accuracy is accompanied by adequate minority-class detection.
- Whether any classifier shows substantially higher variability across folds.

Concrete conclusions will be written after inspecting the experimental
results.

In [30]:
best_f1 = baseline_summary.loc[
    baseline_summary["F1"].idxmax()
]

best_recall = baseline_summary.loc[
    baseline_summary["Recall"].idxmax()
]

best_auc = baseline_summary.loc[
    baseline_summary["AUC"].idxmax()
]

best_brier = baseline_summary.loc[
    baseline_summary["Brier"].idxmin()
]

print("Best F1:")
display(best_f1.to_frame().T.round(4))

print("\nBest Recall:")
display(best_recall.to_frame().T.round(4))

print("\nBest AUC:")
display(best_auc.to_frame().T.round(4))

print("\nBest Brier:")
display(best_brier.to_frame().T.round(4))

Best F1:


,Model,Accuracy,Precision,Recall,Specificity,F1,AUC,Brier
1,Logistic Regression,0.882313,0.734357,0.425906,0.970004,0.537344,0.837485,0.185731



Best Recall:


,Model,Accuracy,Precision,Recall,Specificity,F1,AUC,Brier
0,Mixed Naive Bayes,0.798639,0.420795,0.64529,0.828062,0.508264,0.785102,0.290225



Best AUC:


,Model,Accuracy,Precision,Recall,Specificity,F1,AUC,Brier
1,Logistic Regression,0.882313,0.734357,0.425906,0.970004,0.537344,0.837485,0.185731



Best Brier:


,Model,Accuracy,Precision,Recall,Specificity,F1,AUC,Brier
1,Logistic Regression,0.882313,0.734357,0.425906,0.970004,0.537344,0.837485,0.185731


## 14. Export Baseline Results

The fold-level and aggregated baseline results are stored for later comparison
with the feature-selection experiments.

In [31]:
baseline_fold_results.to_csv(
    "../../results/pa2/baseline_fold_results.csv",
    index=False
)

baseline_results.to_csv(
    "../../results/pa2/baseline_results.csv",
    index=False
)

baseline_summary.to_csv(
    "../../results/pa2/baseline_summary.csv",
    index=False
)

print(
    "PA2 baseline results saved successfully."
)

PA2 baseline results saved successfully.


## 15. Main Observations

Logistic Regression obtains the strongest overall baseline performance. It
achieves the highest mean F1-measure (0.5373), the highest AUC (0.8375), and
the lowest Brier score (0.1857). It also reaches high Precision (0.7344) and
Specificity (0.9700), although its Recall is more moderate (0.4259).

Mixed Naive Bayes provides the highest Recall (0.6453), detecting a larger
proportion of attrition cases than the other classifiers. However, this is
accompanied by lower Precision (0.4208), lower Specificity (0.8281), and a
higher Brier score (0.2902).

LDA obtains good Accuracy (0.8639), Precision (0.6740), Specificity (0.9692),
and AUC (0.8146), but its lower Recall (0.3168) results in a lower F1-measure
(0.4256).

QDA provides the weakest overall baseline results, with the lowest AUC
(0.7211) and F1-measure (0.3946) among the evaluated classifiers.

Overall, Logistic Regression is the most balanced probabilistic baseline,
whereas Mixed Naive Bayes is the most sensitive to the minority attrition
class.